# 机器学习概述（下）：`RunnerV1` 与加州房价回归

上一节我们用线性回归走通了机器学习实践的五要素，但每次重复写「加载 → 拟合 → 评估 → 保存」几行流水代码比较烦琐。本节做两件事：

1. 复用第 1 章末尾实现的 **`RunnerV1`**，把闭式解、评估和模型 IO 串成一条流水线；
2. 跑通**加州房价回归**端到端案例：从 sklearn 拉数据 → 训练/测试划分 → 特征标准化 → 用最小二乘法一步求解 → 评估 → 预测。

> **关于数据集选择**：本节用 sklearn 内置的加州房价数据集（`fetch_california_housing`）。原版书使用的是波士顿房价数据集，但 sklearn 1.2 起已经因伦理问题移除了波士顿数据集，加州房价是目前推荐的回归示范数据集——样本量更大（20 640 个），没有缺失值，特征清晰。

## 1. 准备工作：`Linear` 算子、`optimizer_lsm` 与 `RunnerV1`

这三件东西在第 1 章末尾和第 2 章上节都已经定义过了，现在直接从随书工具包 `nndl` 复用——
等价于把上一章的 inline 代码"打包发布"。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

from pathlib import Path
import torch
from nndl import Linear, optimizer_lsm, RunnerV1

## 2. 加州房价回归

加州房价数据集来自 sklearn 自带的 `fetch_california_housing`：

- 样本数：20 640
- 特征数：8 个（中位收入、房龄、平均房间数、人口、地理位置等）
- 目标：街区房价中位数（单位 \$100k）

实践流程：加载数据 → 训练/测试划分 + 标准化 → 用 `RunnerV1` 求闭式解 → 评估 → 保存与重新加载 → 单点预测。

### 2.1 加载数据

通过 sklearn 直接拉取 DataFrame，避免本地额外的 CSV 文件依赖。

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

ds = fetch_california_housing(as_frame=True)
df = ds.frame                # DataFrame: 8 特征 + 1 target (MedHouseVal)
print('shape:', df.shape)
print('missing per column:', df.isna().sum().sum(), '(no NaN)')
df.head()

**字段说明**：

| 列 | 含义 |
|---|---|
| `MedInc` | 街区中位家庭收入（单位 \$10k） |
| `HouseAge` | 街区房屋中位年龄 |
| `AveRooms` | 平均房间数 |
| `AveBedrms` | 平均卧室数 |
| `Population` | 街区人口 |
| `AveOccup` | 户均人口 |
| `Latitude` / `Longitude` | 街区地理坐标 |
| `MedHouseVal` | **目标**：房价中位数（\$100k） |

从输出看，加州房价数据集没有缺失值，可以直接进入下一步。

### 2.2 数据划分 + 特征标准化

由于度量单位不同，不同特征值的取值范围差别很大（例如 `MedInc` 在 $0\sim15$，`Population` 在 $1\sim35\,000$）。为了消除这种影响，在模型训练前要对特征进行**特征缩放**（feature scaling），让不同特征之间具有可比性。最常见的两种是：

- **归一化**（normalization）：把每一维特征缩放到 $[0, 1]$；
- **标准化**（standardization）：把每一维特征改造为均值 0、方差 1。

这里用标准化。

> **关键约定**：标准化的 `mean/std` **只能在训练集上拟合**，再用同一组统计量去变换测试集——否则会发生测试集信息泄露。

In [ ]:
torch.manual_seed(0)

X = torch.tensor(df.drop(columns=['MedHouseVal']).values, dtype=torch.float32)
y = torch.tensor(df['MedHouseVal'].values, dtype=torch.float32).unsqueeze(1)

# 8:2 划分训练 / 测试集
N = len(X)
perm = torch.randperm(N)
split = int(N * 0.8)
tr, te = perm[:split], perm[split:]
X_train, y_train = X[tr], y[tr]
X_test,  y_test  = X[te], y[te]

# mean/std 只在训练集上拟合
mean, std = X_train.mean(dim=0), X_train.std(dim=0)
X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std

print(f'train: {len(X_train)}  test: {len(X_test)}  feature dim: {X_train.shape[1]}')

### 2.3 用 `RunnerV1` 求闭式解

对线性回归而言，最小二乘法一步就能解出最优参数；模型的评价指标和损失函数一致，都用 MSE。

In [ ]:
def mse(y_true, y_pred):
    return ((y_true - y_pred) ** 2).mean()

model = Linear(input_size=X_train.shape[1])
runner = RunnerV1(model, optimizer_lsm)
runner.fit(X_train, y_train)

print(f'train MSE: {runner.evaluate(X_train, y_train, mse):.4f}')
print(f'test  MSE: {runner.evaluate(X_test,  y_test,  mse):.4f}')

### 2.4 保存与加载

`RunnerV1.save` 把 `model.params` 字典写到磁盘；新建一个同结构模型，再 `load` 回来，应当得到一模一样的预测。

In [ ]:
MODEL_DIR = Path.cwd() / 'models'
save_path = MODEL_DIR / 'california_linear.pt'
runner.save(save_path)

# 新建一个模型，从磁盘加载参数，验证 MSE 与原模型一致
model2 = Linear(input_size=X_train.shape[1])
runner2 = RunnerV1(model2, optimizer_lsm)
runner2.load(save_path)
print(f'reloaded test MSE: {runner2.evaluate(X_test, y_test, mse):.4f}')

### 2.5 检查学到的权重

特征做过标准化之后，权重的正负就能定性解释——正权重表示「该特征增大 → 房价上升」，负权重相反。

In [ ]:
feat_names = list(df.drop(columns=['MedHouseVal']).columns)
weights = model.params['w'].squeeze().tolist()
bias    = model.params['b'].item()

for name, w in sorted(zip(feat_names, weights), key=lambda kv: kv[1]):
    print(f'{name:12s}  {w:+.4f}')
print(f'\nbias         {bias:+.4f}')

**典型观察**：

- `MedInc`（中位收入）权重最正——收入越高的街区房价越高；
- `Latitude` 权重为负，反映加州北部（纬度高）房价低于南部沿海；
- `AveRooms` 权重也呈正——平均房间数大暗示是低密度高端街区。

### 2.6 单点预测

In [ ]:
x0, y0 = X_test[:1], y_test[:1]
pred = runner.predict(x0)
print(f'true price (×$100k): {y0.item():.2f}  →  pred: {pred.item():.2f}')

## 小结

本节复用了第 1 章末尾实现的 `RunnerV1`，端到端跑通了加州房价回归：

- **`RunnerV1`** 把「求闭式解 + 评估 + 模型 IO」封装成一个对象；线性回归只用 `runner.fit(X, y)` 一行就解析地求出最优解；
- **特征标准化** 的统计量只在训练集上拟合，再变换测试集，避免信息泄露；
- **数据集选择**：加州房价是当前 sklearn 推荐的回归示范数据集，波士顿房价已在 sklearn 1.2 中移除。

对线性回归而言，闭式解一步到位、最优解唯一；但对绝大多数非线性模型（Logistic 回归、神经网络等），就没有简单的闭式解了，需要迭代优化。下一章我们引入梯度下降法，并把 `Runner` 升级到 **`RunnerV2`**。